In [ ]:
import os
import base64
import json
import gradio as gr
from openai import OpenAI
from PIL import Image
from io import BytesIO

# --- ARCHITECT'S SETUP ---
api_key = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

# --- THE "ELEGANCE" CSS LAYER ---
custom_css = """
/* Professional Font & Background */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;700;800&display=swap');
body, .gradle-container {
    font-family: 'Inter', sans-serif !important;
    background-color: #f8fafc !important;
}
/* Header Styling */
/* Update this specific block in your custom_css */
#main-header {
    text-align: center;
    padding: 1.5rem 0;
    background: transparent !important; /* Removes the background */
    box-shadow: none !important;        /* Removes the shadow for a flat look */
    color: #1e293b !important;          /* Changed to dark text so it's readable */
    margin-bottom: 1rem;
}
/* Image Preview Styling: Smaller & Centered */
#food-image img {
    max-height: 250px !important;
    border-radius: 12px !important;
    border: 2px solid #e2e8f0 !important;
    margin: 0 auto !important;
}
/* Bold Typography for Labels */
.gr-form label span {
    font-size: 1.1rem !important;
    font-weight: 700 !important;
    color: #1e293b !important;
    text-transform: uppercase;
    letter-spacing: 0.05em;
}
/* Custom Action Button */
#analyze-btn {
    background: #2563eb !important;
    border: none !important;
    font-size: 1.2rem !important;
    font-weight: 800 !important;
    padding: 1rem !important;
    transition: transform 0.2s ease;
}
#analyze-btn:hover {
    transform: translateY(-2px);
    background: #1d4ed8 !important;
}
/* JSON Output Box */
#output-json textarea {
    font-family: 'Fira Code', monospace !important;
    font-size: 1rem !important;
    background-color: #f1f5f9 !important;
    border: 1px solid #cbd5e1 !important;
}
"""

def analyze_food(input_image):
    if input_image is None: return "⚠️ Please upload an image."
    try:
        buffered = BytesIO()
        input_image.save(buffered, format="JPEG")
        base64_img = base64.b64encode(buffered.getvalue()).decode("utf-8")

        prompt = "Analyze this food image. Return ONLY a JSON object with: food_name, calories, protein_grams, fat_grams, and confidence_level."

        response = client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_img}"}}
                ],
            }],
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return {"error": str(e)}

# --- UI CONSTRUCTION (MODERNIZED) ---
# 1. Remove css and theme from here
with gr.Blocks() as demo:
    
    # 2. Replace gr.Div with gr.Column (Div doesn't exist)
    with gr.Column(elem_id="main-header"):
        gr.Markdown("# 🥗 NUTRITION ARCHITECT PRO")
        gr.Markdown("Transforming Vision into Actionable Nutritional Data")

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Group():
                image_input = gr.Image(type="pil", label="1. Upload Food Photo", elem_id="food-image")
                analyze_btn = gr.Button("ANALYZE NUTRITION", variant="primary", elem_id="analyze-btn")
        
        with gr.Column(scale=1):
            with gr.Group():
                json_output = gr.JSON(label="2. Nutritional Intelligence", elem_id="output-json")
                gr.Markdown("> **Architect Note:** Data is synthesized using AI. Accuracy depends on image clarity.")

    analyze_btn.click(fn=analyze_food, inputs=image_input, outputs=json_output)

# 3. ARCHITECT'S MOVE: Pass styling to launch() as per Gradio 6.0 standards
if __name__ == "__main__":
    demo.launch(
        css=custom_css, 
        theme=gr.themes.Default()
    )